# Лабораторная 2. Формирование отчётов в Apache Spark

## Задание
Сформировать отчёт с информацией о 10 наиболее популярных языках программирования по итогам года за период с 2010 по 2020 годы. Отчёт будет отражать динамику изменения популярности языков программирования и представлять собой набор таблиц "топ-10" для каждого года.

Получившийся отчёт сохранить в формате Apache Parquet.

Для выполнения задания вы можете использовать любую комбинацию Spark API: RDD API, Dataset API, SQL API.

# Устанавливаем библиотеку pyspark

In [129]:
!pip -q install pyspark

# Импортируем нужные библиотеки для работы с Apache Spark

In [130]:
import os
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

# Указываем пути к файлам

In [131]:
POSTS_XML = "/content/posts_sample.xml"
LANGS_CSV = "/content/programming-languages.csv"
PARQUET_OUT = "/content/top_languages_2010_2020.parquet"

# Создаем Spark сессию

In [132]:
spark = (
    SparkSession.builder
    .appName("lab2_language_report") # Название приложения
    .master("local[*]") # Запуск на локальной машине с использованием всех доступных ядер
    .getOrCreate()
)
# Устанавливаем уровень логирования, чтобы игнорировать ненужные предупреждения
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 4.0.2


# Загружаем данные о языках программирования из CSV файла

In [133]:
language_source = spark.read.option("header", True).csv(LANGS_CSV)
name_column = language_source.columns[0] # Первый столбец содержит имена языков

# Очищаем данные, приводим названия языков к нижнему регистру и удаляем дубликаты

In [134]:
language_reference = (
    language_source
    .select(F.trim(F.lower(F.col(name_column))).alias("language_name"))
    .where(F.col("language_name").isNotNull()) # Фильтруем пустые строки
    .where(F.col("language_name") != "") # Фильтруем пустые строки
    .dropDuplicates() # Убираем дубликаты
)

print("Количество языков в справочнике:", language_reference.count()) # Выводим количество уникальных языков и первые 10
language_reference.orderBy("language_name").show(10, truncate=False)

Количество языков в справочнике: 698
+-------------+
|language_name|
+-------------+
|@formula     |
|a# (axiom)   |
|a# .net      |
|a+           |
|a++          |
|a-0 system   |
|abap         |
|abc          |
|abc algol    |
|abset        |
+-------------+
only showing top 10 rows


# Загружаем XML файл с постами

In [135]:
raw_rows = spark.read.text(POSTS_XML).withColumnRenamed("value", "raw_xml")

# Фильтруем посты, которые содержат информацию о дате создания и тегах
candidate_posts = (
    raw_rows
    .where(F.col("raw_xml").contains('CreationDate="'))
    .where(F.col("raw_xml").contains('Tags="'))
)
# Извлекаем год и теги из постов
posts_with_year_and_tags = (
    candidate_posts
    .select(
        F.regexp_extract("raw_xml", r'CreationDate="(\d{4})', 1).cast("int").alias("year"),
        F.regexp_extract("raw_xml", r'Tags="([^"]+)"', 1).alias("tag_sequence")
    )
    .where((F.col("year") >= 2010) & (F.col("year") <= 2020))
)

print("Количество постов после фильтрации:", posts_with_year_and_tags.count())
posts_with_year_and_tags.show(20, truncate=False)

Количество постов после фильтрации: 17642
+----+---------------------------------------------------------------------------------------------+
|year|tag_sequence                                                                                 |
+----+---------------------------------------------------------------------------------------------+
|2010|&lt;c++&gt;&lt;character-encoding&gt;                                                        |
|2010|&lt;sharepoint&gt;&lt;infopath&gt;                                                           |
|2010|&lt;iphone&gt;&lt;app-store&gt;&lt;in-app-purchase&gt;                                       |
|2010|&lt;symfony1&gt;&lt;schema&gt;&lt;doctrine&gt;&lt;fixtures&gt;                               |
|2010|&lt;java&gt;                                                                                 |
|2010|&lt;visual-studio-2010&gt;&lt;stylecop&gt;                                                   |
|2010|&lt;cakephp&gt;&lt;file-upload&gt;&lt;swfup

# Нормализуем теги

In [136]:
normalized_tags = (
    posts_with_year_and_tags
    .withColumn("tag_sequence", F.regexp_replace("tag_sequence", r"&lt;", "<"))  # Заменяем &lt; на <
    .withColumn("tag_sequence", F.regexp_replace("tag_sequence", r"&gt;", ">"))  # Заменяем &gt; на >
    .withColumn("tag_sequence", F.regexp_replace("tag_sequence", r"&amp;", "&"))  # Заменяем &amp; на &
    .withColumn("tag_sequence", F.regexp_replace("tag_sequence", r"&quot;", "\""))  # Заменяем &quot; на "
    .withColumn("tag_sequence", F.regexp_replace("tag_sequence", r"&apos;", "'"))  # Заменяем &apos; на '
    .withColumn("tag_sequence", F.regexp_replace("tag_sequence", "<", ""))  # Убираем все угловые скобки <
    .withColumn("tag_sequence", F.regexp_replace("tag_sequence", ">", " "))  # Заменяем > на пробел
    .withColumn("single_tag", F.explode(F.split(F.trim(F.col("tag_sequence")), r"\s+")))  # Разбиваем строку на отдельные теги
    .select(
        "year",
        F.trim(F.lower(F.col("single_tag"))).alias("language_name")  # Приводим к нижнему регистру
    )
    .where(F.col("language_name") != "")  # Фильтруем пустые строки
)
print("Количество постов после фильтрации:", normalized_tags.count())
normalized_tags.show(20, truncate=False)


Количество постов после фильтрации: 52118
+----+------------------+
|year|language_name     |
+----+------------------+
|2010|c++               |
|2010|character-encoding|
|2010|sharepoint        |
|2010|infopath          |
|2010|iphone            |
|2010|app-store         |
|2010|in-app-purchase   |
|2010|symfony1          |
|2010|schema            |
|2010|doctrine          |
|2010|fixtures          |
|2010|java              |
|2010|visual-studio-2010|
|2010|stylecop          |
|2010|cakephp           |
|2010|file-upload       |
|2010|swfupload         |
|2010|git               |
|2010|cygwin            |
|2010|putty             |
+----+------------------+
only showing top 20 rows


# SQL запрос для подсчета упоминаний языков программирования

In [137]:
normalized_tags.createOrReplaceTempView("tag_mentions") # Создаем временные представления для SQL запросов
language_reference.createOrReplaceTempView("language_reference")

language_mentions = spark.sql("""
    SELECT
        m.year,
        m.language_name,
        COUNT(*) AS mentions
    FROM tag_mentions m
    INNER JOIN language_reference l
        ON m.language_name = l.language_name
    GROUP BY m.year, m.language_name
""")

language_mentions.orderBy("year", F.desc("mentions")).show(20, truncate=False) # Сортируем данные по году и количеству упоминаний

+----+-------------+--------+
|year|language_name|mentions|
+----+-------------+--------+
|2010|java         |52      |
|2010|php          |46      |
|2010|javascript   |44      |
|2010|python       |26      |
|2010|objective-c  |23      |
|2010|c            |20      |
|2010|ruby         |12      |
|2010|delphi       |8       |
|2010|r            |3       |
|2010|applescript  |3       |
|2010|perl         |3       |
|2010|bash         |3       |
|2010|haskell      |2       |
|2010|sed          |2       |
|2010|f#           |2       |
|2010|xpath        |1       |
|2010|racket       |1       |
|2010|actionscript |1       |
|2010|mouse        |1       |
|2010|scheme       |1       |
+----+-------------+--------+
only showing top 20 rows


# Топ-10 языков по годам

In [138]:
ranking_window = Window.partitionBy("year").orderBy(F.desc("mentions"), F.asc("language_name")) # Добавляем ранжирование языков по годам

top_languages_long = (
    language_mentions
    .withColumn("rank", F.row_number().over(ranking_window)) # Присваиваем ранг каждому языку
    .where(F.col("rank") <= 10)  # Оставляем только 10
    .select("year", "rank", "language_name", "mentions")
    .orderBy("year", "rank")
)

top_languages_long.orderBy("year", "rank").show(100, truncate=False)

+----+----+-------------+--------+
|year|rank|language_name|mentions|
+----+----+-------------+--------+
|2010|1   |java         |52      |
|2010|2   |php          |46      |
|2010|3   |javascript   |44      |
|2010|4   |python       |26      |
|2010|5   |objective-c  |23      |
|2010|6   |c            |20      |
|2010|7   |ruby         |12      |
|2010|8   |delphi       |8       |
|2010|9   |applescript  |3       |
|2010|10  |bash         |3       |
|2011|1   |php          |102     |
|2011|2   |java         |93      |
|2011|3   |javascript   |83      |
|2011|4   |python       |37      |
|2011|5   |objective-c  |34      |
|2011|6   |c            |24      |
|2011|7   |ruby         |20      |
|2011|8   |perl         |9       |
|2011|9   |delphi       |8       |
|2011|10  |bash         |7       |
|2012|1   |php          |154     |
|2012|2   |javascript   |132     |
|2012|3   |java         |124     |
|2012|4   |python       |69      |
|2012|5   |objective-c  |45      |
|2012|6   |c        

# Переводим данные в широкоформатную таблицу

In [139]:
wide_report = (
    top_languages_long
    .groupBy("year")
    .pivot("rank", list(range(1, 11)))
    .agg(F.first("language_name"))
)

for idx in range(1, 11):
    wide_report = wide_report.withColumnRenamed(str(idx), f"top_{idx}") # Переименовываем столбцы для лучшего представления

wide_report = wide_report.orderBy("year") # Сортируем данные по году
wide_report.show(truncate=False)

+----+----------+----------+----------+------+-----------+-----------+-----------+-----------+-----------+------+
|year|top_1     |top_2     |top_3     |top_4 |top_5      |top_6      |top_7      |top_8      |top_9      |top_10|
+----+----------+----------+----------+------+-----------+-----------+-----------+-----------+-----------+------+
|2010|java      |php       |javascript|python|objective-c|c          |ruby       |delphi     |applescript|bash  |
|2011|php       |java      |javascript|python|objective-c|c          |ruby       |perl       |delphi     |bash  |
|2012|php       |javascript|java      |python|objective-c|c          |ruby       |bash       |r          |lua   |
|2013|javascript|php       |java      |python|objective-c|c          |ruby       |r          |bash       |scala |
|2014|javascript|java      |php       |python|c          |objective-c|r          |ruby       |bash       |matlab|
|2015|javascript|java      |php       |python|r          |c          |objective-c|ruby  

# Сохраняем результат в формате Parquet

In [140]:
(top_languages_long
    .write
    .mode("overwrite")
    .parquet(PARQUET_OUT))

print("Результат сохранён в:", PARQUET_OUT)

Результат сохранён в: /content/top_languages_2010_2020.parquet


# Загружаем сохранённый файл и показываем результат

In [141]:
restored = spark.read.parquet(PARQUET_OUT)
restored.orderBy("year", "rank").show(200, truncate=False)
print("Количество строк в restored:", restored.count())
restored.select("year").distinct().orderBy("year").show(20, False) # Уникальные года

+----+----+-------------+--------+
|year|rank|language_name|mentions|
+----+----+-------------+--------+
|2010|1   |java         |52      |
|2010|2   |php          |46      |
|2010|3   |javascript   |44      |
|2010|4   |python       |26      |
|2010|5   |objective-c  |23      |
|2010|6   |c            |20      |
|2010|7   |ruby         |12      |
|2010|8   |delphi       |8       |
|2010|9   |applescript  |3       |
|2010|10  |bash         |3       |
|2011|1   |php          |102     |
|2011|2   |java         |93      |
|2011|3   |javascript   |83      |
|2011|4   |python       |37      |
|2011|5   |objective-c  |34      |
|2011|6   |c            |24      |
|2011|7   |ruby         |20      |
|2011|8   |perl         |9       |
|2011|9   |delphi       |8       |
|2011|10  |bash         |7       |
|2012|1   |php          |154     |
|2012|2   |javascript   |132     |
|2012|3   |java         |124     |
|2012|4   |python       |69      |
|2012|5   |objective-c  |45      |
|2012|6   |c        


# Завершаем работу с Spark

In [142]:
spark.stop()

# Архивируем файл с результатами

In [143]:
import shutil

# архивируем parquet папку
shutil.make_archive(
    "top_languages_2010_2020", # Имя архива
    'zip', # Формат архива
    "/content/top_languages_2010_2020.parquet" # Путь к директории для архивации
)

'/content/top_languages_2010_2020.zip'

# Скачиваем архив

In [144]:
from google.colab import files
files.download("top_languages_2010_2020.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>